# Section 3 - Workflows, Part 1: Sequential & Concurrent

## Our Use Case

You might wonder: *"Customer wants to increase his pension contribution on NL-2031-887. What should I say and what compliance guardrails apply?"* This is **not a single-agent task**. It's a pipeline:

1. **Triage** the rep request into the right service category.
2. **Advise** on the pension-specific customer-service answer.
3. **Check compliance** against DNB / AFM expectations before the rep uses it.

We *could* shove all three into one prompt, but each step has different requirements (factuality vs. empathy vs. structured output). Workflows let us compose specialised agents into a deterministic pipeline.

## What you'll build

| Pattern | Athora Netherlands use-case | When to choose it |
|---------|------------------|-------------------|
| **Sequential** | TriageAgent → PensionAdvisorAgent → ComplianceCheckerAgent | Steps depend on each other's output. |
| **Concurrent** (fan-out / fan-in) | Same rep query, reviewed in parallel by *Product*, *Risk*, and *Compliance* perspectives | Independent perspectives, fastest wall-clock. |

Both use the workflow builders `SequentialBuilder` and `ConcurrentBuilder`. Reference: [`microsoft/agent-framework — python/samples/03-workflows`](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows).

## Setup — shared Foundry client

We re-use the Foundry chat client. Each step in the workflow is just an `Agent` with focused instructions.

In [5]:
import os
from agent_framework import Agent, Message
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from typing import cast

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=credential,
)
print("Foundry client ready.")

Foundry client ready.


### Introduction to Workflows in Agent Framework



## Sequential workflow: triage → pension advice → compliance check

We define three small agents. Each only does its slice. `SequentialBuilder` chains them so each agent's output becomes the next agent's input.

In [6]:
triage_agent = Agent(
    client=client,
    name="TriageAgent",
    instructions=(
        "Classify Athora Netherlands rep requests as PENSION, LIFE_INSURANCE, CLAIM, COMPLAINT, or OTHER. "
        "Return the category and the policy number if present."
    ),
)

pension_advisor_agent = Agent(
    client=client,
    name="PensionAdvisorAgent",
    instructions=(
        "Help Athora Netherlands reps explain defined-contribution pension options. "
        "Use only the facts provided by earlier steps; do not give regulated financial advice."
    ),
)

compliance_checker_agent = Agent(
    client=client,
    name="ComplianceCheckerAgent",
    instructions=(
        "Review the draft for DNB / AFM guardrails. Flag missing disclaimers, suitability checks, "
        "or unsupported advice. End with APPROVED or NEEDS_REVIEW."
    ),
)

sequential_workflow = SequentialBuilder(
    participants=[triage_agent, pension_advisor_agent, compliance_checker_agent]
).build()
print("Sequential workflow built.")

Sequential workflow built.


In [7]:
from agent_framework import AgentResponseUpdate

REP_QUERY = (
    "Jan de Vries is on the line about pension policy NL-2031-887. "
    "He currently contributes EUR 350 per month and asks whether increasing it is possible. "
    "Draft guidance for the rep, including compliance guardrails."
)

# In streaming mode, output events carry AgentResponseUpdate chunks from the LAST agent only.
updates: list[AgentResponseUpdate] = []
async for event in sequential_workflow.run(REP_QUERY, stream=True):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        updates.append(event.data)

print("===== ComplianceCheckerAgent (final agent) response =====\n")
final_text = "".join(u.text for u in updates)
print(final_text)

===== ComplianceCheckerAgent (final agent) response =====

NEEDS_REVIEW

**Flagged Issues:**

- **Missing Disclaimers:**  
  - There is no explicit disclaimer stating that this is not financial advice and that the caller should consult an independent (tax or financial) advisor for personal suitability or tax implications.
- **Suitability Checks:**  
  - No mention of asking the client about his current circumstances, objectives, or potential employer restrictions before discussing increasing contributions.
- **Unsupported Advice:**  
  - The draft shares general, factual information but does not cross into giving personal recommendations. However, to be fully compliant, it must avoid even the suggestion that increasing a contribution is advisable, unless all suitability questions and assessments are completed or the caller is referred to a registered advisor.

**Recommendations:**
- Add explicit disclaimers about the nature of the information (not advice).
- Suggest a suitability and a

### What just happened

- The rep query arrived as the first item in the shared conversation.
- `TriageAgent` classified it as a pension request and preserved the policy number.
- `PensionAdvisorAgent` produced rep guidance from the triage context.
- `ComplianceCheckerAgent` reviewed the guidance against regulators' guardrails.
- Every step was an ordinary `Agent` — no special wrapper. The `SequentialBuilder` only added the wiring.

### Try it in DevUI

Try running this sample in DevUI:

```bash
python devui/sequential_devui.py
```

This opens the full Triage → PensionAdvisor → ComplianceChecker pipeline`. Try it with the existing rep queries or test your own!

## Concurrent workflow: parallel review of one rep query

For some Athora Netherlands processes, the *same* input needs **independent** reviews. Example: a product-change question should be reviewed in parallel by:

- **product** — what options are available
- **risk** — whether risk-profile guardrails apply
- **compliance** — regularors' wording and documentation concerns

Each gets the same input and writes its own opinion. `ConcurrentBuilder` fans out, then aggregates.

In [8]:
from agent_framework import AgentResponse

compliance = Agent(
    client=client,
    name="compliance",
    instructions=(
        "You are an Athora Netherlands compliance reviewer. Flag DNB / AFM, KYC, or documentation concerns "
        "in the rep query. 3 bullets max."
    ),
)

risk = Agent(
    client=client,
    name="risk",
    instructions=(
        "You are a risk reviewer. Flag anything unusual about the requested change, "
        "risk profile, or product fit. 3 bullets max."
    ),
)

product = Agent(
    client=client,
    name="product",
    instructions=(
        "You are a product specialist. Explain what information the rep needs before discussing changes. "
        "3 bullets max."
    ),
)

review_workflow = ConcurrentBuilder(participants=[compliance, risk, product]).build()

PRODUCT_CHANGE = (
    "Rep query: Sanne Bakker has term life policy NL-4408-552 with monthly premium EUR 42.50. "
    "She asks whether changing cover or premium is possible after a life-event update. "
    "What should the rep check before responding?"
)

# Non-streaming: get_outputs() returns AgentResponse objects with .messages
events = await review_workflow.run(PRODUCT_CHANGE)
for output in events.get_outputs():
    response = cast(AgentResponse, output)
    for msg in response.messages:
        name = msg.author_name or "user"
        print(f"\n=== {name} ===\n{msg.text}")


=== compliance ===
- **DNB/AFM Concern:** Ensure that any advice given complies with "suitability" and "duty of care" (zorgplicht) requirements under Wft, as changing coverage or premium may require an updated needs analysis.
- **KYC Concern:** Confirm that recent KYC (customer identification, verification, and life-event documentation) is up-to-date, as changes after life events may necessitate a client file review.
- **Documentation Concern:** Check if there is sufficient documentation of the life event (e.g., marriage, birth, divorce), and that it is correctly recorded in the client file before proceeding with policy changes.

=== risk ===
- **Policy Terms & Conditions:** Confirm if the policy allows changes to coverage or premium after inception, specifically following a life event (e.g., marriage, childbirth, home purchase). Some term life policies are locked after issue or only permit changes within certain windows or under qualifying events.

- **Evidence of Life Event:** Check

### What just happened

- The same `PRODUCT_CHANGE` text was fanned out to all three agents in parallel.
- Each agent saw **only the original input**, not each other's output.
- The default aggregator stitched their outputs into one combined `list[Message]`.
- Wall-clock time ≈ slowest agent, not the sum.

If you wanted the three reviews to **see each other's verdict** before producing a final decision, that's a *sequential-after-concurrent* pattern — built by combining the two builders.

### Try it in DevUI

Try running this sample in DevUI:

```bash
python devui/concurrent_devui.py
```

This opens the full Triage → PensionAdvisor → ComplianceChecker pipeline`. Try it with the existing rep queries or test your own!

## Design notes

Sequential workflows fit when every answer must pass through a known set of gates. Concurrent workflows fit when several independent perspectives can be gathered side-by-side, then summarized for a rep. In production, Athora Netherlands would usually combine them: concurrent specialist reviews followed by a sequential compliance summarizer.

## Recap and what's next

- **Sequential** = each step builds on the previous; shared `list[Message]` carries context.
- **Concurrent** = same input, parallel perspectives, default fan-in aggregator.
- Both reuse the **same `Agent` objects** you built in Sections 1-2 — workflows are composition, not a different agent model.

**Coming up in Section 4:** what if the routing isn't fixed? PolicyPal needs to *decide at runtime* whether to hand off to a `ClaimsAgent` or a `RiskAgent`. That's **handoff** orchestration, plus a peek at **Magentic** for fully open-ended tasks.